# 2장 1강: AARRR 프레임워크 개요와 단계별 분석 목적 — 실습문제

## 실습 목표

- Ravenstack의 사용자 여정을 AARRR 5단계로 구분할 수 있다.
- 각 단계의 분석 목적과 활용 가능한 데이터를 연결할 수 있다.
- 이벤트 로그의 사용자·행동·시점·속성 컬럼을 구분할 수 있다.
- 분석 질문에 맞게 퍼널·코호트·세그멘테이션 분석을 선택할 수 있다.
- 그룹별 Activation 지표를 비교하여 점검 대상을 찾을 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`
- `ravenstack_feature_usage.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

Ravenstack의 AARRR 단계는 다음과 같이 연결할 수 있습니다.

| 단계 | Ravenstack에서의 의미 | 활용 데이터 예시 |
|---|---|---|
| Acquisition | 고객사가 유입되어 가입함 | 가입일, 유입 경로 |
| Activation | 구독 후 제품 기능을 처음 사용함 | 기능 사용 기록 |
| Retention | 제품 사용과 구독을 계속 유지함 | 반복 사용 기록, 이탈 여부 |
| Referral | 기존 고객이 다른 고객을 추천함 | 추천 전송·추천 가입 이벤트 |
| Revenue | 유료 구독에서 반복 매출이 발생함 | MRR, 요금제 |

> 제공된 데이터에는 고객 추천 행동을 직접 기록한 이벤트가 없습니다. `referral_source`는 고객의 유입 경로이며, 기존 고객이 추천 행동을 했다는 사실을 직접 보여주는 Referral 이벤트와는 구분해야 합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 세 CSV 파일을 각각 `accounts`, `subscriptions`, `feature_usage`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`, `usage_date`를 날짜형으로 변환하세요.
6. `feature_usage`에서 이벤트 식별자인 `usage_id`의 중복 개수를 확인하세요.
7. 기능별 이벤트 기록 수와 `usage_count`의 기초 통계량을 확인하세요.

In [2]:
# 실습 준비 코드를 작성하세요.
# 1. 라이브러리 불러오기
import pandas as pd

pd.set_option('display.max_columns', None)

# 2. 세 CSV 파일 불러오기
accounts = pd.read_csv('ravenstack_accounts.csv')
subscriptions = pd.read_csv('ravenstack_subscriptions.csv')
feature_usage = pd.read_csv('ravenstack_feature_usage.csv')

tables = {'accounts': accounts, 'subscriptions': subscriptions, 'feature_usage': feature_usage}

# 3. 행·열 개수와 상위 5개 행 확인
for name, df in tables.items():
    print(f'[{name}] 행 {df.shape[0]:,}개 / 열 {df.shape[1]}개')
    display(df.head())

# 4. 자료형과 결측치 개수 확인
for name, df in tables.items():
    print(f'[{name}] 자료형 · 결측치')
    display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'null_count': df.isna().sum()}))

# 5. 날짜 컬럼을 날짜형으로 변환
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])   # 결측(NaN) → NaT: 아직 종료되지 않은 구독
feature_usage['usage_date'] = pd.to_datetime(feature_usage['usage_date'])

print(accounts[['signup_date']].dtypes, subscriptions[['start_date', 'end_date']].dtypes,
      feature_usage[['usage_date']].dtypes, sep='\n')
print('usage_date 범위:', feature_usage['usage_date'].min().date(), '~', feature_usage['usage_date'].max().date())

# 6. usage_id 중복 개수 확인
dup_cnt = feature_usage['usage_id'].duplicated().sum()
print(f'usage_id 중복 개수: {dup_cnt}')
print(f'행 전체가 동일한 완전 중복 개수: {feature_usage.duplicated().sum()}')
# → usage_id는 겹치지만 구독·날짜·기능이 모두 다른 행이므로 '같은 이벤트의 중복 적재'가 아닌 ID 충돌로 보고 제거하지 않음
display(feature_usage[feature_usage['usage_id'].duplicated(keep=False)].sort_values('usage_id').head(6))

# 7. 기능별 이벤트 기록 수와 usage_count 기초 통계량
print('기능 종류 수:', feature_usage['feature_name'].nunique())
display(feature_usage['feature_name'].value_counts().to_frame('event_count').head(10))
display(feature_usage['usage_count'].describe().to_frame())


[accounts] 행 500개 / 열 10개


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


[subscriptions] 행 5,000개 / 열 14개


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


[feature_usage] 행 25,000개 / 열 8개


,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,False


... (7 outputs left out)

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
19294,U-0c9318,S-01b2dc,2023-12-11,feature_9,5,1060,3,False
17533,U-0c9318,S-0ffab0,2024-01-30,feature_11,3,1614,0,False
20588,U-13ce5b,S-9b623b,2023-03-26,feature_28,10,2050,1,False
9626,U-13ce5b,S-8b0950,2024-09-10,feature_9,8,824,0,True
18480,U-2103bb,S-7fc49b,2023-04-18,feature_12,9,5022,1,False
10379,U-2103bb,S-ae3270,2024-01-06,feature_3,10,5510,0,False


기능 종류 수: 40


,event_count
feature_name,
feature_12,659
feature_32,659
feature_6,655
feature_17,651
feature_34,650
feature_26,649
feature_36,648
feature_31,644
feature_20,643


,usage_count
count,25000.000000
mean,10.021000
std,3.143729
min,0.000000
25%,8.000000
50%,10.000000
75%,12.000000
max,26.000000


---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 결제 주기별 Activation 지표 비교하기

#### 문제 설명

프로덕트팀은 2024년 12월 제품 활성률이 월간 결제 구독과 연간 결제 구독에서 다르게 나타나는지 확인하려고 합니다.

> 이 과제는 필수 2의 요금제별 세그멘테이션을 `billing_frequency`에 동일하게 적용하는 문제입니다.

#### 요구사항

1. 필수 2에서 만든 `eligible_dec`과 `is_product_active`를 사용하세요.
2. `billing_frequency`별로 다음 값을 집계하여 `billing_activation`을 만드세요.
   - 이용 가능 유료 구독 수
   - 제품 활성 유료 구독 수
   - 제품 활성률
3. 제품 활성률이 낮은 순서로 정렬하고 백분율로 출력하세요.
4. 제품 활성률이 더 낮은 결제 주기를 찾으세요.
5. 해당 그룹을 우선 점검 대상으로 정하고 추가로 확인할 내용을 한 가지 제안하세요.
6. 이 분석의 AARRR 단계와 분석 기법을 설명하세요.

#### 해석 질문

**Q1.** 제품 활성률이 더 낮은 결제 주기는 무엇인가요?  
**Q2.** 이 분석은 어떤 AARRR 단계와 분석 기법에 해당하나요?  
**Q3.** 두 그룹의 제품 활성률 차이만으로 결제 주기가 낮은 활성의 원인이라고 결론 내릴 수 있나요?

**Q4.** 퍼널 분석·코호트 리텐션 분석·세그멘테이션을 표로 정리하세요. 각 분석의 계산 또는 분류 원리, 해결하려는 분석 질문, 관련 AARRR 단계를 설명하세요. 전환율·이탈률의 분모, 코호트의 시작 기준과 경과 기간, 세그먼트 분류 기준을 포함하세요.

#### 제출 결과

- `billing_activation` 집계 코드와 결과
- 우선 점검할 결제 주기
- 추가 확인 내용
- AARRR 단계와 분석 기법
- Q1~Q4 답변

In [5]:
dec_start = pd.Timestamp('2024-12-01')
dec_end = pd.Timestamp('2024-12-31')

# 12월 이용 가능 유료 구독: 체험 아님 + 12월 말 이전 시작 + (종료일 없음 or 12월 1일 이후 종료)
eligible_dec = subscriptions[
    (subscriptions['is_trial'] == False)
    & (subscriptions['start_date'] <= dec_end)
    & (subscriptions['end_date'].isna() | (subscriptions['end_date'] >= dec_start))
].copy()

# 12월에 기능을 사용한 구독 ID
dec_usage = feature_usage[
    (feature_usage['usage_date'] >= dec_start) & (feature_usage['usage_date'] <= dec_end)
]
active_subscription_ids = dec_usage['subscription_id'].unique()

eligible_dec['is_product_active'] = eligible_dec['subscription_id'].isin(active_subscription_ids)

# ── 과제 1. 결제 주기별 Activation 비교 ──
# 2. billing_frequency별 집계
billing_activation = eligible_dec.groupby('billing_frequency').agg(
    eligible_subscriptions=('subscription_id', 'nunique'),
    active_subscriptions=('is_product_active', 'sum')
)
billing_activation['activation_rate'] = (
    billing_activation['active_subscriptions'] / billing_activation['eligible_subscriptions']
)

# 3. 제품 활성률 낮은 순 정렬 + 백분율 출력
billing_activation = billing_activation.sort_values('activation_rate')

print('[결제 주기별 2024년 12월 제품 활성률]')
for freq, row in billing_activation.iterrows():
    print(f"{freq:>8} | 이용 가능 {row['eligible_subscriptions']:,}개 | "
          f"제품 활성 {row['active_subscriptions']:,}개 | 활성률 {row['activation_rate']:.2%}")

# 4. 제품 활성률이 더 낮은 결제 주기
lowest = billing_activation.index[0]
gap = billing_activation['activation_rate'].iloc[-1] - billing_activation['activation_rate'].iloc[0]
print(f'\n활성률이 더 낮은 결제 주기: {lowest} ({billing_activation.loc[lowest, "activation_rate"]:.2%})')
print(f'두 그룹 차이: {gap * 100:.2f}%p')

# 5. 추가 확인: 요금제 × 결제 주기별 제품 활성률
plan_billing = pd.crosstab(
    eligible_dec['billing_frequency'], eligible_dec['plan_tier'],
    values=eligible_dec['is_product_active'], aggfunc='mean'
)
print('\n[요금제 × 결제 주기별 제품 활성률 (%)]')
print((plan_billing * 100).round(1))

[결제 주기별 2024년 12월 제품 활성률]
  annual | 이용 가능 1,968.0개 | 제품 활성 385.0개 | 활성률 19.56%
 monthly | 이용 가능 2,013.0개 | 제품 활성 403.0개 | 활성률 20.02%

활성률이 더 낮은 결제 주기: annual (19.56%)
두 그룹 차이: 0.46%p

[요금제 × 결제 주기별 제품 활성률 (%)]
plan_tier          Basic  Enterprise   Pro
billing_frequency                         
annual              18.9        20.3  19.4
monthly             20.0        20.6  19.4


### 과제 1 답변 작성란

**결과 요약**: 2024년 12월 이용 가능 유료 구독 3,981개 중 788개가 제품 기능을 사용했습니다.
결제 주기별 제품 활성률은 annual 19.56%(385/1,968), monthly 20.02%(403/2,013)입니다.

- **우선 점검할 결제 주기**: annual (연간 결제)
- **추가 확인 내용**: 연간 결제 구독의 12월 이전 월별 기능 사용 추이를 확인하여, 계약 이후 시간이 지날수록 사용이 줄어드는지(구독은 유지되지만 실제 사용은 멈춘 구독이 많은지) 살펴봅니다.
- **AARRR 단계와 분석 기법**: Activation / 세그멘테이션

- **Q1.** annual(연간 결제)입니다. annual의 제품 활성률은 19.56%로 monthly(20.02%)보다 약 0.46%p 낮습니다.

- **Q2.** 유료 구독이 12월에 제품 기능을 실제로 사용하여 핵심 가치를 경험했는지를 보므로 **Activation** 단계입니다. 구독을 결제 주기(`billing_frequency`)라는 특성 기준으로 나누어 같은 지표(제품 활성률)를 비교했으므로 **세그멘테이션 분석**입니다.

- **Q3.** 결론 내릴 수 없습니다.
  1. 두 그룹의 차이가 0.46%p로 매우 작아 우연한 변동일 수 있습니다.
  2. 세그멘테이션은 "어떤 집단에서 차이가 나타나는지"를 보여줄 뿐, 그 특성이 원인임을 증명하지 않습니다. 요금제, 좌석 수, 산업, 구독 시작 시점 등 다른 요인이 두 그룹에 다르게 분포하면 그 요인이 차이를 만들었을 수 있습니다.
  3. 따라서 "annual 구독의 활성률이 낮다"는 것은 **확인된 현상**이고, "결제 주기 때문에 활성이 낮다"는 것은 **검증이 필요한 가설**입니다. 추가 분석과 실험으로 검증해야 합니다.

- **Q4.**

| 분석 기법 | 계산 또는 분류 원리 | 해결하려는 분석 질문 | 관련 AARRR 단계 |
|---|---|---|---|
| **퍼널 분석** | 목표 행동까지의 과정을 단계로 나누고 각 단계 도달 사용자 수를 셈. **구간별 전환율 = 다음 단계 사용자 수 ÷ 현재 단계 사용자 수 × 100** (분모: 현재 단계 도달 사용자). **이탈률 = 100% − 전환율** (분모 동일). **전체 전환율 = 최종 단계 사용자 수 ÷ 최초 단계 사용자 수 × 100** | 목표 행동까지 가는 과정 중 어느 구간에서 다음 단계로 이어지지 않는가? (문제가 나타나는 **행동 구간**) | Acquisition → Activation, Activation → Revenue 등 단계 간 전환 |
| **코호트 리텐션 분석** | 같은 **시작 기준**(가입일·첫 구매일·첫 사용일 등)을 공유하는 사용자를 코호트로 묶고, 시작 시점을 Day 0으로 맞춘 뒤 **경과 기간**(Day 1/7/30, Week 1/2, Month 1/2 등)마다 다시 활동한 비율을 계산. **리텐션율 = 특정 시점 재활동 사용자 수 ÷ 코호트 최초 사용자 수 × 100** (모든 시점의 분모는 최초 코호트 인원) | 시작 이후 시간이 지나도 계속 이용하는가? 어느 시점·어느 코호트부터 유지율이 달라지는가? (문제가 나타나는 **시점**) | Retention |
| **세그멘테이션** | 사용자를 특정 **특성이나 행동 기준**으로 여러 집단으로 나누고 같은 지표를 비교. 분류 기준 예: 사용자 특성(신규/기존), 유입 경로(검색/광고/추천), 사용 환경(모바일/PC), 이용 행동(빈도 높음/낮음), 상품·서비스(무료/유료, 요금제, 결제 주기) | 어떤 사용자 집단에서 문제가 나타나는가? (문제가 집중되는 **사용자**) | 모든 단계에 적용 가능 (이번 과제는 Activation) |

## 실습 마무리

1. **Ravenstack의 사용자 여정을 어떤 AARRR 단계로 나누었나요?**
   - **Acquisition**: 고객사가 유입되어 가입함 (가입일, 유입 경로 `referral_source`)
   - **Activation**: 구독 후 제품 기능을 처음 사용함 (기능 사용 기록)
   - **Retention**: 제품 사용과 구독을 계속 유지함 (반복 사용 기록, 이탈 여부 `churn_flag`)
   - **Referral**: 기존 고객이 다른 고객을 추천함 (추천 전송·추천 가입 이벤트 — 현재 데이터에 없음)
   - **Revenue**: 유료 구독에서 반복 매출이 발생함 (MRR, 요금제)

2. **각 단계는 어떤 분석 질문에 답하기 위해 사용했나요?**
   - Acquisition: 고객사가 얼마나, 어떤 경로로 들어왔는가? → 가입 고객사 500개
   - Activation: 구독한 고객이 제품의 핵심 가치를 실제로 경험했는가? → 기능 사용 기록이 있는 구독 비율 99.34%
   - Retention: 고객이 구독을 계속 유지하고 있는가? → 미이탈 구독 비율 90.28% (정식 리텐션율이 아닌 대리 지표)
   - Referral: 기존 고객이 새 고객을 데려오는가? → 직접 측정 불가
   - Revenue: 사용이 실제 반복 매출로 이어지는가? → 현재 활성 유료 구독 MRR 합계 10,159,608

3. **제공된 데이터에서 직접 측정하기 어려운 단계는 무엇이었나요?**
   - **Referral**입니다. `referral_source`는 고객이 어떤 경로로 유입되었는지를 나타낼 뿐, 기존 고객이 추천 행동을 했다는 사실을 기록한 이벤트가 아닙니다. 직접 측정하려면 추천 링크 생성·전송, 추천 링크 클릭, 추천을 통한 가입(추천인 ID 포함) 같은 이벤트가 필요합니다.
   - 덧붙여 **Retention**도 `churn_flag`로는 간단한 대리 지표만 볼 수 있습니다. 정식 리텐션은 코호트 기준과 경과 기간을 정해 다시 사용한 비율로 계산해야 합니다.

4. **요금제와 결제 주기별 비교에는 어떤 분석 기법을 사용했나요?**
   - **세그멘테이션 분석**입니다. 2024년 12월 이용 가능 유료 구독을 `plan_tier`(요금제)와 `billing_frequency`(결제 주기)라는 속성 기준으로 나누고, 같은 지표(제품 활성률 = 12월 제품 활성 유료 구독 수 ÷ 12월 이용 가능 유료 구독 수)를 비교해 어떤 집단에서 Activation이 낮은지 확인했습니다.

5. **분석 결과를 근거로 어떤 대상을 우선 점검했나요?**
   - 요금제별: **Pro** 요금제(19.37%)가 가장 낮았습니다. (Basic 19.49%, Enterprise 20.48%)
   - 결제 주기별: **annual(연간 결제)**(19.56%)이 monthly(20.02%)보다 낮았습니다.
   - 두 그룹을 우선 점검 대상으로 정했습니다. 다만 그룹 간 차이가 1%p 안팎으로 작습니다. 따라서 "해당 그룹의 활성률이 낮다"는 것은 확인된 현상일 뿐이고, 요금제나 결제 주기가 원인이라고 단정할 수 없습니다. 월별 사용 추이, 사용 기능 종류, 오류 발생 수(`error_count`) 등을 추가로 확인해 가설을 세우고 검증해야 합니다.